In [ ]:
import os
from medcat.cat import CAT
from medcat import cat
import pandas as pd
import json
from tqdm.notebook import tqdm
import re
import pickle

In [ ]:
# Add file logger
import logging
medcat_logger = logging.getLogger('medcat')
fh = logging.FileHandler('20260203_medcat.log')
medcat_logger.addHandler(fh)

# Paths and Config

In [ ]:
# relative path to working_with_cogstack folder
_rel_path = os.path.join("..", "..", "..", "..")
# absolute path to working_with_cogstack folder
base_path = os.path.abspath(_rel_path)

project_path = base_path + '/projects/M-46-G_chronic_kidney_disease/'
data_path = project_path + 'data/'

In [ ]:
# Changes these according to your project
project_name = '20260203_ckd_comorbidities_search'

## Clinical Notes
### Original Search
#clinical_noting_docs = data_path+'raw_data/elasticsearch_search_hits/20250714_clinical_noting_comorbidities_search.csv'
### Data Refresh
#clinical_noting_docs = data_path+'raw_data/elasticsearch_search_hits/20251120_clinical_noting_comorbidities_search.csv'
### Updated Inclusion Patients
clinical_noting_docs = data_path+'raw_data/elasticsearch_search_hits/20260202_clinical_noting_comorbidities_search.csv'

## EPR Letters
### Original Search
#epr_letters = data_path+'raw_data/elasticsearch_search_hits/20250714_epr_comorbidities_search.csv'
### Data Refresh
#epr_letters = data_path+'raw_data/elasticsearch_search_hits/20251120_epr_comorbidities_search.csv'
### Updated Inclusion Patients
epr_letters = data_path+'raw_data/elasticsearch_search_hits/20260202_epr_comorbidities_search.csv'

## EPIC Notes
### Original Search
#epic_notes = data_path+'raw_data/elasticsearch_search_hits/20250714_epic_comorbidities_search.csv'
### Data Refresh
#epic_notes = data_path+'raw_data/elasticsearch_search_hits/20251120_epic_comorbidities_search.csv'
### Updated Inclusion Patients
epic_notes = data_path+'raw_data/elasticsearch_search_hits/20260202_epic_comorbidities_search.csv'

modelpack = 'medcat_model.zip'  # enter your model here. Should the the output of trained 'output_modelpack'.
snomed_filter_path = data_path+'raw_data/project_filter/20241018_m_40_G_snomed_concept.json'

# Constants (nothing to change below)
doc_id_column = "_id"
doc_text_column = "document_Content"

model_dir = 'models/20241120_trained_ckd_model'
model_pack_path = os.path.join(base_path, model_dir, modelpack)

ann_folder_path = data_path+'processed_data/ann_folder_path/'+project_name
if not os.path.exists(ann_folder_path):
    os.makedirs(ann_folder_path)
    print(f'Created folder to store annotations here: {ann_folder_path}')
    
save_path_annotations_per_doc = os.path.join(base_path, ann_folder_path, "all_ckd_comorbidity_annotations.json")

# Load MedCAT model

In [ ]:
# Create CAT - the main class from medcat used for concept annotation
cat = CAT.load_model_pack(model_pack_path)

# Annotate

In [ ]:
# Set snomed filter if needed
# This is a white list filter of concepts
if snomed_filter_path:
    snomed_filter = set(json.load(open(snomed_filter_path)))
else:
    print('There is no concept filter set')
    snomed_filter = set(cat.cdb.cui2preferred_name.keys())

cat.config.linking['filters']['cuis'] = snomed_filter

In [ ]:
clinic_notes = pd.read_csv(clinical_noting_docs, dtype=object)[[doc_id_column, doc_text_column]]

In [ ]:
epr_letters = pd.read_csv(epr_letters, dtype=object)[[doc_id_column, doc_text_column]]

In [ ]:
epic_notes = pd.read_csv(epic_notes, dtype=object)[[doc_id_column, doc_text_column]]

In [ ]:
df = pd.concat([clinic_notes, epr_letters, epic_notes])  # Not necessary to filter at this step. But this loads only what is required
df = df.drop_duplicates().reset_index(drop=True)

del clinic_notes, epr_letters, epic_notes

df.shape

In [ ]:
try:
    annotation_chunks = os.listdir(ann_folder_path)
    annotation_chunks.remove('annotated_ids.pickle')
    
    annotations = {}
    for file in annotation_chunks:
        with open(os.path.join(ann_folder_path, file), 'rb') as pkl_file:
                  annotations.update(pickle.load(pkl_file))
    
    annotated_docs = []
    
    for doc, anns in annotations.items():
        annotated_docs.append(doc)
    
    annotated_docs = list(set(annotated_docs))
    
    print(len(annotated_docs))
except:
    annotated_docs = None

In [ ]:
if annotated_docs is not None:
    df = df[~df['_id'].isin(annotated_docs)]

df.shape

In [ ]:
# Create generator object
def data_iterator(data, doc_name, doc_text):
    for id, row in data.iterrows():
        yield (row[doc_name], row[doc_text])

In [ ]:
batch_char_size = 50000  # Batch size (BS) in number of characters
cat.multiprocessing_batch_char_size(data_iterator(df, doc_id_column, doc_text_column),
                                    batch_size_chars=batch_char_size,
                                    only_cui=False,
                                    nproc=8, # Number of processors
                                    out_split_size_chars=20*batch_char_size,
                                    save_dir_path=ann_folder_path,
                                    min_free_memory=0.0001,
                                    )

medcat_logger.warning(f'Annotation process complete!')

### Double check if everything has been annotated.

This does not check meta-annotations

In [ ]:
# Check if everything has run smoothly. If an error has been raised check the logs
try:
    # Path to your pickle file
    pickle_file_path = os.path.join(ann_folder_path, "annotated_ids.pickle")
    # Open the pickle file in read mode
    with open(pickle_file_path, "rb") as pickle_file:
        loaded_data = pickle.load(pickle_file)
    assert len(df) == len(loaded_data[0])
except AssertionError as e:
    print("Error:", "There are documents which havent been annotated! Check 'medcat.log' for more info")


END OF SCRIPT